# Adobe Leadership Agent Demo

This notebook is the fastest way to review the assignment.

How to use it:

1. Run the import cell.
2. Edit the configuration cell if you want to set an API key, change models, or switch parsers.
3. Run the helper cell.
4. Edit `QUESTION` if needed, then run the final cell.

Notes:

- If `OPENAI_API_KEY` is left blank, the notebook runs the deterministic offline answer path.
- The first dense-index build may take longer because `fastembed` may download the local embedding model.
- Answer JSON files are saved under `outputs/sample_answers/`.


In [ ]:
from pathlib import Path
import json

from IPython.display import Markdown, display

from leadership_agent.answering import LeadershipAgent, render_report, save_report
from leadership_agent.config import AppConfig
from leadership_agent.utils import slugify

PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / "config.yaml"

print(f"Project root: {PROJECT_ROOT}")
print(f"Config path: {CONFIG_PATH}")


In [ ]:
# Edit these values if you want to test with your own key or model choices.

OPENAI_API_KEY = ""
OPENAI_NANO_MODEL = "gpt-5.4-nano"
OPENAI_MINI_MODEL = "gpt-5.4-mini"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

PARSER_PRIMARY = "docling"
PARSER_FALLBACK = "marker"

FORCE_REBUILD = False
QUESTION = "What is our current revenue trend?"
REPORT_FILENAME = None  # Example: "review_answer.json"

SAMPLE_QUESTIONS = [
    "What is our current revenue trend?",
    "Which departments are underperforming?",
    "What were the key risks highlighted in the last quarter?",
]

SAMPLE_QUESTIONS


In [ ]:
def build_notebook_config() -> AppConfig:
    config = AppConfig.load(CONFIG_PATH)
    config.models.nano = OPENAI_NANO_MODEL.strip() or config.models.nano
    config.models.mini = OPENAI_MINI_MODEL.strip() or config.models.mini
    config.models.embedding = EMBEDDING_MODEL.strip() or config.models.embedding
    config.parser.primary = PARSER_PRIMARY.strip() or config.parser.primary
    config.parser.fallback = PARSER_FALLBACK.strip() or config.parser.fallback
    if OPENAI_API_KEY.strip():
        config.runtime.llm_provider = "openai"
        config.runtime.openai_api_key = OPENAI_API_KEY.strip()
    else:
        config.runtime.llm_provider = "extractive"
        config.runtime.openai_api_key = ""
    return config


def parsed_ready(config: AppConfig) -> bool:
    required = [
        config.parsed_data_dir / "sections" / "sections.jsonl",
        config.parsed_data_dir / "chunks" / "chunks.jsonl",
        config.parsed_data_dir / "tables" / "tables.jsonl",
    ]
    return all(path.exists() for path in required)


def indexes_ready(config: AppConfig) -> bool:
    required = [
        config.index_dir / "bm25" / "sections.pkl",
        config.index_dir / "bm25" / "chunks.pkl",
        config.index_dir / "dense" / "sections.meta.json",
        config.index_dir / "metadata" / "leadership.duckdb",
    ]
    return all(path.exists() for path in required)


def prepare_pipeline(config: AppConfig, force_rebuild: bool = False):
    agent = LeadershipAgent(config)
    status = {}
    if force_rebuild or not parsed_ready(config):
        status["ingest"] = agent.ingest()
    else:
        status["ingest"] = "skipped"
    if force_rebuild or not indexes_ready(config):
        status["build_index"] = agent.build_index()
    else:
        status["build_index"] = "skipped"
    return agent, status


def ask_notebook_question(question: str, report_filename: str | None = None, force_rebuild: bool = False):
    config = build_notebook_config()
    agent, prep_status = prepare_pipeline(config, force_rebuild=force_rebuild)
    report = agent.ask(question, output_dir=config.output_dir)

    filename = (report_filename or f"{slugify(question)}.json").strip()
    if not filename.lower().endswith(".json"):
        filename += ".json"
    report_path = config.output_dir / "sample_answers" / filename
    save_report(report, report_path)

    summary = {
        "question": question,
        "llm_provider": config.runtime.llm_provider,
        "nano_model": config.models.nano,
        "mini_model": config.models.mini,
        "embedding_model": config.models.embedding,
        "parser_primary": config.parser.primary,
        "parser_fallback": config.parser.fallback,
        "report_path": str(report_path),
    }

    print("Run summary:")
    print(json.dumps(summary, indent=2))
    print("\nPreparation status:")
    print(json.dumps(prep_status, indent=2))
    print("\nRendered answer:\n")
    display(Markdown(render_report(report)))
    return report, report_path, prep_status


In [ ]:
report, report_path, prep_status = ask_notebook_question(
    question=QUESTION,
    report_filename=REPORT_FILENAME,
    force_rebuild=FORCE_REBUILD,
)

report_path